# Persisted, Reusable Models: Standardized Ridge vs. Interpretable OLS

Two things this notebook adds on top of `1.5-cc-national-lag-vs-current-food-insecurity-model.ipynb`:

1. **Persistence.** Every fitted model (features, means/stds, coefficients) is saved to a
   JSON file under `models/cc/` via `src/models/train_model.save_model`. Scoring a future
   year's data no longer requires re-running or rebuilding this notebook -- just
   `src/models/predict_model.load_model` + `forecast`, using the shared feature-engineering
   code in `src/features/build_features.py`.
2. **An interpretable, non-standardized variant.** Alongside the standardized ridge models
   from 1.5 (coefficients only meaningful in "per standard deviation" terms), each spec is
   also fit as a plain population-weighted OLS on raw-unit features (no standardization, no
   ridge penalty) -- coefficients are then directly interpretable ("per 1-unit change in this
   driver"). We compare accuracy between the two to see whether the easier-to-read version
   costs anything.

All feature engineering, model fitting, and metrics logic now lives in `src/`
(`src/data/make_dataset.py`, `src/features/build_features.py`, `src/models/train_model.py`,
`src/models/predict_model.py`) rather than being copied inline into the notebook, so it's
importable from anywhere, not just this notebook.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve project paths so the notebook works from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "external").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.make_dataset import clean_columns, read_sheet_from_workbooks
from src.features.build_features import DRIVER_COLS, LAG_SPECS, build_panel, build_transition_table
from src.models.train_model import evaluate_lag_spec, save_model, weighted_metrics
from src.models.predict_model import forecast, load_model

## Configuration

In [2]:
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models" / "cc"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MMG_WORKBOOK_PATHS = [
    EXTERNAL_DIR / "MMG_2019-2023.xlsx",
    EXTERNAL_DIR / "MMG_2024.xlsx",
]
ALICE_COUNTY_PATH = EXTERNAL_DIR / "2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv"

TARGET_STATE = "FL"
TARGET_FOOD_BANK = "Feeding Tampa Bay"

OUTPUT_PATH = PROCESSED_DIR / "lag_vs_current_ridge_vs_ols_comparison.csv"

# Short filename slugs for each spec, used when saving model artifacts.
SPEC_SLUGS = {
    "current (X[t], rate[t-1])": "current",
    "2-year lag (X[t-2], rate[t-2])": "lag2",
    "3-year lag (X[t-3], rate[t-3])": "lag3",
}

MMG_WORKBOOK_PATHS, ALICE_COUNTY_PATH, MODELS_DIR, OUTPUT_PATH

([WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/MMG_2019-2023.xlsx'),
  WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/MMG_2024.xlsx')],
 WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv'),
 WindowsPath('C:/Users/Tom/ftb-pro-bono/models/cc'),
 WindowsPath('C:/Users/Tom/ftb-pro-bono/data/processed/lag_vs_current_ridge_vs_ols_comparison.csv'))

## Load Source Data and Build the Panel

In [3]:
zcta_raw = read_sheet_from_workbooks(MMG_WORKBOOK_PATHS, "ZCTA")
county_raw = read_sheet_from_workbooks(MMG_WORKBOOK_PATHS, "County")
alice_county_raw = clean_columns(pd.read_csv(ALICE_COUNTY_PATH, dtype=str))

panel = build_panel(zcta_raw, county_raw, alice_county_raw)

observed_years = sorted(panel["year"].dropna().unique().tolist())
print(f"Observed years: {observed_years}")
print(f"Rows: {len(panel):,}, unique row_ids: {panel['row_id'].nunique():,}")
print(f"Florida rows: {(panel['State'] == TARGET_STATE).sum():,}")

Observed years: [2020, 2021, 2022, 2023, 2024]
Rows: 232,268, unique row_ids: 46,947
Florida rows: 5,837


## Fit and Persist Both Variants of Each Spec

For each of the three lag specs, fit both the standardized ridge model (as in 1.5) and the
raw-unit OLS model, then save each to `models/cc/`.

In [4]:
results = {}
for label, (driver_lag, rate_lag) in LAG_SPECS.items():
    for standardize in (True, False):
        result = evaluate_lag_spec(
            panel=panel,
            label=label,
            driver_lag=driver_lag,
            rate_lag=rate_lag,
            driver_cols=DRIVER_COLS,
            target_state=TARGET_STATE,
            target_food_bank=TARGET_FOOD_BANK,
            standardize=standardize,
        )
        variant = "ridge" if standardize else "ols"
        results[(label, variant)] = result

        model_path = MODELS_DIR / f"{SPEC_SLUGS[label]}_{variant}.json"
        save_model(result, model_path)

        print(
            f"{label} [{variant}]: method={result['validation_method']}, alpha={result['selected_alpha']:g}, "
            f"fl_mae={result['fl_weighted_mae']:.4f} -> saved to {model_path.relative_to(PROJECT_ROOT)}"
        )

current (X[t], rate[t-1]) [ridge]: method=time_holdout, alpha=2, fl_mae=0.0160 -> saved to models\cc\current_ridge.json


current (X[t], rate[t-1]) [ols]: method=time_holdout, alpha=0, fl_mae=0.0160 -> saved to models\cc\current_ols.json


2-year lag (X[t-2], rate[t-2]) [ridge]: method=time_holdout, alpha=2, fl_mae=0.0276 -> saved to models\cc\lag2_ridge.json


2-year lag (X[t-2], rate[t-2]) [ols]: method=time_holdout, alpha=0, fl_mae=0.0276 -> saved to models\cc\lag2_ols.json


3-year lag (X[t-3], rate[t-3]) [ridge]: method=time_holdout, alpha=2, fl_mae=0.0179 -> saved to models\cc\lag3_ridge.json


3-year lag (X[t-3], rate[t-3]) [ols]: method=time_holdout, alpha=0, fl_mae=0.0237 -> saved to models\cc\lag3_ols.json


## Compare Ridge vs. OLS Accuracy

In [5]:
comparison_df = pd.DataFrame([
    {
        "specification": label,
        "variant": variant,
        "validation_method": r["validation_method"],
        "selected_alpha": r["selected_alpha"],
        "n_train": r["n_train"],
        "n_test": r["n_test"],
        "national_weighted_mae": r["national_weighted_mae"],
        "fl_weighted_mae": r["fl_weighted_mae"],
        "fl_weighted_rmse": r["fl_weighted_rmse"],
        "fl_weighted_mean_error": r["fl_weighted_mean_error"],
        "ftb_weighted_mae": r["ftb_weighted_mae"],
    }
    for (label, variant), r in results.items()
])
comparison_df = comparison_df.sort_values(["specification", "variant"]).reset_index(drop=True)
display(comparison_df)
comparison_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved comparison table to: {OUTPUT_PATH}")

,specification,variant,validation_method,selected_alpha,n_train,n_test,national_weighted_mae,fl_weighted_mae,fl_weighted_rmse,fl_weighted_mean_error,ftb_weighted_mae
0,"2-year lag (X[t-2], rate[t-2])",ols,time_holdout,0.0,72217,36369,0.027125,0.027610,0.031310,0.023903,0.026498
1,"2-year lag (X[t-2], rate[t-2])",ridge,time_holdout,2.0,72217,36369,0.027125,0.027610,0.031310,0.023903,0.026498
2,"3-year lag (X[t-3], rate[t-3])",ols,time_holdout,0.0,35922,36035,0.018513,0.023709,0.029924,-0.017540,0.020014
3,"3-year lag (X[t-3], rate[t-3])",ridge,time_holdout,2.0,35922,36035,0.020726,0.017870,0.024768,-0.010949,0.014811
4,"current (X[t], rate[t-1])",ols,time_holdout,0.0,109148,36591,0.015114,0.015985,0.021628,0.001413,0.010772
5,"current (X[t], rate[t-1])",ridge,time_holdout,2.0,109148,36591,0.015114,0.015985,0.021628,0.001413,0.010772


Saved comparison table to: C:\Users\Tom\ftb-pro-bono\data\processed\lag_vs_current_ridge_vs_ols_comparison.csv


## Interpretable (OLS) Coefficients, Raw Units

Unlike the standardized ridge coefficients, these are directly interpretable: each one is
"predicted food insecurity rate changes by this many percentage points per 1-unit increase
in this driver, holding the others fixed." (Most drivers here are already rates between 0 and
1, so "1 unit" is a 100-percentage-point swing -- divide by 100 to read it as "per 1 percentage
point" for those.)

In [6]:
ols_coef_frame = pd.DataFrame({"feature": ["intercept"] + results[(list(LAG_SPECS)[0], "ols")]["feature_cols"]})
for label in LAG_SPECS:
    r = results[(label, "ols")]
    ols_coef_frame[label] = r["model"]["coef"]
display(ols_coef_frame)

,feature,"current (X[t], rate[t-1])","2-year lag (X[t-2], rate[t-2])","3-year lag (X[t-3], rate[t-3])"
0,intercept,-1.657329e-01,2.367104e-01,3.229421e-11
1,lag_food_insecurity_rate,8.468467e-01,6.372434e-01,6.568181e-01
2,unemployment_rate,7.479985e-02,3.290136e-02,-2.276313e-02
3,poverty_rate,9.660016e-02,6.386473e-02,3.654649e-02
4,percent_black,-4.435614e-03,-4.140521e-03,8.867072e-03
5,percent_hispanic,9.402053e-03,2.276540e-02,3.079414e-02
6,log_median_income,1.486484e-02,-1.356612e-02,-1.194867e-02
7,homeownership_rate,-1.596068e-02,-2.183168e-02,-3.146073e-02
8,disability_rate,7.167459e-02,6.031024e-02,5.607054e-02
9,county_child_population_share,-2.652006e-02,-2.880252e-02,-3.488628e-02


## Verify Persistence Round-Trip

Reload the saved 3-year-lag OLS model from disk (a fresh dict, not the in-memory one from
above) and re-score the same held-out year, to confirm the persisted artifact reproduces the
same result -- this is the check that scoring a future year from a saved model actually works,
not just that fitting in-notebook works.

In [7]:
from src.models.train_model import predict_weighted_ridge
from src.features.build_features import clip_rate

check_label = "3-year lag (X[t-3], rate[t-3])"
check_variant = "ols"
driver_lag, rate_lag = LAG_SPECS[check_label]

reloaded = load_model(MODELS_DIR / f"{SPEC_SLUGS[check_label]}_{check_variant}.json")

table = build_transition_table(panel, driver_lag, rate_lag, DRIVER_COLS)
valid = table.dropna(subset=["food_insecurity_rate", "lag_food_insecurity_rate", "population"] + DRIVER_COLS).copy()
latest_year = valid["year"].max()
test = valid.loc[valid["year"] == latest_year].copy()
test["is_fl"] = test["State"].astype(str).str.upper() == TARGET_STATE

test["prediction"] = clip_rate(predict_weighted_ridge(reloaded, test[reloaded["feature_cols"]]))
reloaded_fl_mae = weighted_metrics(test.loc[test["is_fl"]], "food_insecurity_rate", "prediction")["weighted_mae"]
in_memory_fl_mae = results[(check_label, check_variant)]["fl_weighted_mae"]

print(f"In-memory fitted FL MAE:  {in_memory_fl_mae:.6f}")
print(f"Reloaded-from-disk FL MAE: {reloaded_fl_mae:.6f}")
assert np.isclose(reloaded_fl_mae, in_memory_fl_mae), "Persisted model did not reproduce the in-memory fit."
print("Match -- the persisted artifact reproduces the original fit exactly.")

In-memory fitted FL MAE:  0.023709
Reloaded-from-disk FL MAE: 0.023709
Match -- the persisted artifact reproduces the original fit exactly.


## Forecast Beyond Observed Data

This is the actual payoff: using only data already observed (2024, the latest year in the
panel), the persisted 2-year-lag model forecasts **2026**, and the persisted 3-year-lag model
forecasts **2027** -- both years we have no ground truth for yet. When 2025 data arrives later
this year, calling `forecast(model, panel, source_year=2025)` on the 3-year-lag model would
forecast 2028 the same way, with no notebook changes required.

In [8]:
lag2_model = load_model(MODELS_DIR / "lag2_ridge.json")
lag3_model = load_model(MODELS_DIR / "lag3_ridge.json")

forecast_2026 = forecast(lag2_model, panel, source_year=2024)
forecast_2027 = forecast(lag3_model, panel, source_year=2024)

print("2026 forecast (2-year-lag model, Florida rows):")
display(forecast_2026.loc[forecast_2026["State"] == TARGET_STATE].sort_values("population", ascending=False).head(10))

print("2027 forecast (3-year-lag model, Florida rows):")
display(forecast_2027.loc[forecast_2027["State"] == TARGET_STATE].sort_values("population", ascending=False).head(10))

2026 forecast (2-year-lag model, Florida rows):


,row_id,State,Food Bank 1,population,forecast_year,prediction
30515,12_12069_34787,FL,Second Harvest Food Bank of Central Florida,101441,2026,0.141356
32035,12_12095_34787,FL,Second Harvest Food Bank of Central Florida,101441,2026,0.141457
33191,12_12111_34953,FL,Treasure Coast Food Bank,86875,2026,0.148215
28582,12_12011_33025,FL,Feeding South Florida,76967,2026,0.183233
32170,12_12099_33411,FL,Feeding South Florida,76863,2026,0.135957
28577,12_12011_33024,FL,Feeding South Florida,76585,2026,0.194035
33151,12_12109_32259,FL,Feeding Northeast Florida,75016,2026,0.095237
29472,12_12031_32259,FL,Feeding Northeast Florida,75016,2026,0.094242
28697,12_12011_33311,FL,Feeding South Florida,74898,2026,0.200584
30194,12_12057_33647,FL,Feeding Tampa Bay,74844,2026,0.147758


2027 forecast (3-year-lag model, Florida rows):


,row_id,State,Food Bank 1,population,forecast_year,prediction
30515,12_12069_34787,FL,Second Harvest Food Bank of Central Florida,101441,2027,0.132030
32035,12_12095_34787,FL,Second Harvest Food Bank of Central Florida,101441,2027,0.131285
33191,12_12111_34953,FL,Treasure Coast Food Bank,86875,2027,0.138921
28582,12_12011_33025,FL,Feeding South Florida,76967,2027,0.179417
32170,12_12099_33411,FL,Feeding South Florida,76863,2027,0.125121
28577,12_12011_33024,FL,Feeding South Florida,76585,2027,0.185843
33151,12_12109_32259,FL,Feeding Northeast Florida,75016,2027,0.080931
29472,12_12031_32259,FL,Feeding Northeast Florida,75016,2027,0.082845
28697,12_12011_33311,FL,Feeding South Florida,74898,2027,0.192358
30194,12_12057_33647,FL,Feeding Tampa Bay,74844,2027,0.139874


## Notes and Next Steps

- All six fitted models (3 specs x {ridge, ols}) are saved under `models/cc/` as JSON --
  features, means/stds, coefficients, and the metrics/metadata they were fit with.
- Compare the `fl_weighted_mae` column above between `ridge` and `ols` rows for the same
  specification: if OLS is close to (or better than) ridge, the more interpretable version
  can replace ridge going forward with little accuracy cost.
- To score a brand-new year once it's available: read its MMG/ALICE files with
  `read_sheet_from_workbooks` / `pd.read_csv`, call `build_panel(...)` (same function used
  here), then `forecast(loaded_model, new_panel, source_year=<new year>)`. No notebook
  rebuild required -- but see the caveat below.
- **A frozen model is not permanent.** These coefficients reflect the driver/outcome
  relationships observed in 2020-2024. Applying them to a future source year assumes those
  relationships still hold; the recommended practice is to periodically refit (e.g. whenever
  a new year of MMG data lands) rather than trust one saved model indefinitely -- the scoring
  *code* stays the same either way, only the saved coefficients get refreshed.